# Fruit Classification — Neural Network Training Pipeline
Exhaustive deep learning pipeline: architecture search, regularisation, callbacks, fine-tuning, quantisation-aware export.

## 1. Imports & Configuration

In [ ]:
import os
import json
import warnings
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, f1_score
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, mixed_precision
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint,
    CSVLogger, TensorBoard, LambdaCallback
)
import tensorflow_model_optimization as tfmot

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams['figure.dpi'] = 120

FEATURES   = ['r', 'g', 'b']
FRUITS_DIR = 'fruits'          # folder with per-class CSVs
# FRUITS_CSV = 'fruits.csv'    # uncomment if using single-file layout

CHECKPOINT_DIR = 'checkpoints'
LOG_DIR        = 'logs'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print("TF version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices('GPU'))

## 2. Data Loading & Preprocessing

In [ ]:
frames = []
for fname in sorted(os.listdir(FRUITS_DIR)):
    if not fname.endswith('.csv'):
        continue
    label = os.path.splitext(fname)[0]
    tmp = pd.read_csv(os.path.join(FRUITS_DIR, fname))
    tmp.columns = tmp.columns.str.strip().str.lower()
    tmp['fruit'] = label
    frames.append(tmp)

df = pd.concat(frames, ignore_index=True)
df.drop_duplicates(inplace=True)
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

# remove 3σ outliers
for col in FEATURES:
    z = (df[col] - df[col].mean()) / df[col].std()
    df = df[z.abs() <= 3]
df.reset_index(drop=True, inplace=True)

print(f"Dataset shape: {df.shape}")
print(df['fruit'].value_counts())

### 2.1 Encode & scale

In [ ]:
le = LabelEncoder()
y_int = le.fit_transform(df['fruit'].values)
class_names = list(le.classes_)
n_classes   = len(class_names)
n_features  = len(FEATURES)

X_raw = df[FEATURES].values.astype(np.float32)

scaler = MinMaxScaler()
X = scaler.fit_transform(X_raw).astype(np.float32)

# one-hot targets for softmax
Y = keras.utils.to_categorical(y_int, num_classes=n_classes)

print("X shape:", X.shape, "  Y shape:", Y.shape)
print("Classes:", class_names)

### 2.2 Train / validation / test split

In [ ]:
X_tv, X_test, Y_tv, Y_test, yi_tv, yi_test = train_test_split(
    X, Y, y_int, test_size=0.15, random_state=SEED, stratify=y_int
)
X_train, X_val, Y_train, Y_val, yi_train, yi_val = train_test_split(
    X_tv, Y_tv, yi_tv, test_size=0.15, random_state=SEED, stratify=yi_tv
)

print(f"Train : {X_train.shape[0]}")
print(f"Val   : {X_val.shape[0]}")
print(f"Test  : {X_test.shape[0]}")

### 2.3 Class-weight computation

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

cw_vals = compute_class_weight('balanced', classes=np.arange(n_classes), y=yi_train)
class_weights = dict(enumerate(cw_vals))
print("Class weights:", {class_names[k]: round(v, 4) for k, v in class_weights.items()})

### 2.4 tf.data pipeline

In [ ]:
BATCH_SIZE = 32
AUTOTUNE   = tf.data.AUTOTUNE

def add_noise(x, y):
    """Gaussian noise augmentation on raw features."""
    noise = tf.random.normal(shape=tf.shape(x), mean=0.0, stddev=0.015, seed=SEED)
    return tf.clip_by_value(x + noise, 0.0, 1.0), y

ds_train = (
    tf.data.Dataset.from_tensor_slices((X_train, Y_train))
    .shuffle(buffer_size=512, seed=SEED)
    .map(add_noise, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
ds_val = (
    tf.data.Dataset.from_tensor_slices((X_val, Y_val))
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
ds_test = (
    tf.data.Dataset.from_tensor_slices((X_test, Y_test))
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print("Pipeline ready.")

## 3. Helper Utilities

In [ ]:
def plot_history(history, title='Training History'):
    hist = history.history
    epochs = range(1, len(hist['loss']) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].plot(epochs, hist['loss'],     label='Train loss',     linewidth=1.8)
    axes[0].plot(epochs, hist['val_loss'], label='Val loss',       linewidth=1.8, linestyle='--')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Loss')
    axes[0].legend()

    axes[1].plot(epochs, hist['accuracy'],     label='Train acc', linewidth=1.8)
    axes[1].plot(epochs, hist['val_accuracy'], label='Val acc',   linewidth=1.8, linestyle='--')
    axes[1].yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1))
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Accuracy')
    axes[1].legend()

    fig.suptitle(title, fontweight='bold')
    sns.despine()
    plt.tight_layout()
    plt.show()


def evaluate_model(model, X_te, y_int_te, title=''):
    y_prob = model.predict(X_te, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    acc  = accuracy_score(y_int_te, y_pred)
    f1   = f1_score(y_int_te, y_pred, average='macro')
    print(f"{title}")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  F1 Macro : {f1:.4f}")
    print()
    print(classification_report(y_int_te, y_pred, target_names=class_names))

    cm = confusion_matrix(y_int_te, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'Confusion Matrix — {title}')
    plt.tight_layout()
    plt.show()
    return acc, f1, y_pred, y_prob


def count_params(model):
    total     = model.count_params()
    trainable = sum(np.prod(v.shape) for v in model.trainable_weights)
    print(f"  Total params     : {total:,}")
    print(f"  Trainable params : {trainable:,}")
    return total, trainable

## 4. Baseline MLP

In [ ]:
def build_baseline(n_features, n_classes):
    inp = keras.Input(shape=(n_features,), name='rgb_input')
    x   = layers.Dense(16, activation='relu', name='hidden1')(inp)
    x   = layers.Dense(8,  activation='relu', name='hidden2')(x)
    out = layers.Dense(n_classes, activation='softmax', name='output')(x)
    return keras.Model(inp, out, name='baseline_mlp')

baseline = build_baseline(n_features, n_classes)
baseline.summary()
count_params(baseline)

In [ ]:
baseline.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_base = [
    EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6, verbose=1),
    ModelCheckpoint(
        filepath=os.path.join(CHECKPOINT_DIR, 'baseline_best.keras'),
        monitor='val_loss', save_best_only=True, verbose=0
    ),
    CSVLogger(os.path.join(LOG_DIR, 'baseline.csv'))
]

history_base = baseline.fit(
    ds_train,
    validation_data=ds_val,
    epochs=200,
    class_weight=class_weights,
    callbacks=callbacks_base,
    verbose=0
)

plot_history(history_base, 'Baseline MLP')

In [ ]:
base_acc, base_f1, _, _ = evaluate_model(baseline, X_test, yi_test, title='Baseline MLP')

## 5. Regularised Deep MLP

In [ ]:
def build_deep_mlp(n_features, n_classes, units=(64, 32, 16),
                   dropout=0.3, l2=1e-4, batch_norm=True):
    inp = keras.Input(shape=(n_features,), name='rgb_input')
    x = inp
    for i, u in enumerate(units):
        x = layers.Dense(
            u, use_bias=not batch_norm,
            kernel_regularizer=regularizers.l2(l2),
            name=f'dense_{i}'
        )(x)
        if batch_norm:
            x = layers.BatchNormalization(name=f'bn_{i}')(x)
        x = layers.Activation('relu', name=f'act_{i}')(x)
        x = layers.Dropout(dropout, seed=SEED, name=f'drop_{i}')(x)
    out = layers.Dense(n_classes, activation='softmax', name='output')(x)
    return keras.Model(inp, out, name='deep_mlp')


deep = build_deep_mlp(n_features, n_classes)
deep.summary()
count_params(deep)

In [ ]:
deep.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_deep = [
    EarlyStopping(monitor='val_loss', patience=30, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=12, min_lr=1e-7, verbose=1),
    ModelCheckpoint(
        filepath=os.path.join(CHECKPOINT_DIR, 'deep_best.keras'),
        monitor='val_accuracy', save_best_only=True, verbose=0
    ),
    CSVLogger(os.path.join(LOG_DIR, 'deep.csv'))
]

history_deep = deep.fit(
    ds_train,
    validation_data=ds_val,
    epochs=300,
    class_weight=class_weights,
    callbacks=callbacks_deep,
    verbose=0
)

plot_history(history_deep, 'Regularised Deep MLP')

In [ ]:
deep_acc, deep_f1, _, _ = evaluate_model(deep, X_test, yi_test, title='Deep MLP')

## 6. Residual MLP (skip connections)

In [ ]:
def residual_block(x, units, l2=1e-4, dropout=0.2):
    shortcut = x
    x = layers.Dense(units, kernel_regularizer=regularizers.l2(l2), use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(dropout, seed=SEED)(x)
    x = layers.Dense(units, kernel_regularizer=regularizers.l2(l2), use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    if shortcut.shape[-1] != units:
        shortcut = layers.Dense(units, use_bias=False)(shortcut)
    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    return x


def build_residual_mlp(n_features, n_classes):
    inp = keras.Input(shape=(n_features,))
    x   = layers.Dense(32, activation='relu')(inp)
    x   = residual_block(x, 32)
    x   = residual_block(x, 32)
    x   = layers.GlobalAveragePooling1D()(layers.Reshape((32, 1))(x))
    out = layers.Dense(n_classes, activation='softmax')(x)
    return keras.Model(inp, out, name='residual_mlp')


res_model = build_residual_mlp(n_features, n_classes)
res_model.summary()
count_params(res_model)

In [ ]:
res_model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=5e-4, weight_decay=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_res = res_model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=300,
    class_weight=class_weights,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=30, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=12, min_lr=1e-7, verbose=1),
        ModelCheckpoint(
            filepath=os.path.join(CHECKPOINT_DIR, 'residual_best.keras'),
            monitor='val_accuracy', save_best_only=True, verbose=0
        ),
        CSVLogger(os.path.join(LOG_DIR, 'residual.csv'))
    ],
    verbose=0
)

plot_history(history_res, 'Residual MLP')

In [ ]:
res_acc, res_f1, _, _ = evaluate_model(res_model, X_test, yi_test, title='Residual MLP')

## 7. Learning Rate Schedulers

In [ ]:
# Visualise all LR schedules before committing to one

def cosine_decay_with_warmup(epoch, lr, warmup=10, total=200):
    if epoch < warmup:
        return lr * (epoch + 1) / warmup
    progress = (epoch - warmup) / (total - warmup)
    return 5e-4 * 0.5 * (1 + np.cos(np.pi * progress))

def step_decay(epoch, lr):
    drop_rate, epochs_drop = 0.5, 50
    return 1e-3 * (drop_rate ** (epoch // epochs_drop))

epochs_range = np.arange(200)
cosine_lrs   = [cosine_decay_with_warmup(e, 0, warmup=10, total=200) for e in epochs_range]
step_lrs     = [step_decay(e, 1e-3) for e in epochs_range]

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(epochs_range, cosine_lrs, label='Cosine Decay + Warmup', linewidth=1.8)
ax.plot(epochs_range, step_lrs,   label='Step Decay',            linewidth=1.8, linestyle='--')
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_title('LR Schedules')
ax.legend()
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Train deep MLP with cosine schedule as ablation
cosine_model = build_deep_mlp(n_features, n_classes)
cosine_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cosine_cb = keras.callbacks.LearningRateScheduler(
    lambda e, lr: cosine_decay_with_warmup(e, lr, warmup=10, total=300)
)

history_cosine = cosine_model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=300,
    class_weight=class_weights,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=40, restore_best_weights=True, verbose=1),
        cosine_cb,
        CSVLogger(os.path.join(LOG_DIR, 'cosine.csv'))
    ],
    verbose=0
)

plot_history(history_cosine, 'Deep MLP + Cosine LR')
cosine_acc, cosine_f1, _, _ = evaluate_model(cosine_model, X_test, yi_test, title='Cosine LR')

## 8. Manual Architecture Grid Search

In [ ]:
grid = {
    'units':    [(32, 16), (64, 32), (64, 32, 16), (128, 64, 32)],
    'dropout':  [0.2, 0.3],
    'l2':       [1e-4, 1e-3],
    'batch_norm': [True, False]
}

grid_results = []

for units, dropout, l2, bn in itertools.product(
        grid['units'], grid['dropout'], grid['l2'], grid['batch_norm']):

    m = build_deep_mlp(n_features, n_classes,
                       units=units, dropout=dropout, l2=l2, batch_norm=bn)
    m.compile(
        optimizer=keras.optimizers.Adam(5e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    h = m.fit(
        ds_train,
        validation_data=ds_val,
        epochs=150,
        class_weight=class_weights,
        callbacks=[EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)],
        verbose=0
    )
    val_acc = max(h.history['val_accuracy'])
    y_pred  = np.argmax(m.predict(X_test, verbose=0), axis=1)
    te_f1   = f1_score(yi_test, y_pred, average='macro')

    grid_results.append({
        'units': str(units), 'dropout': dropout, 'l2': l2,
        'batch_norm': bn, 'val_acc': round(val_acc, 4), 'test_f1': round(te_f1, 4)
    })
    print(f"units={units} dr={dropout} l2={l2} bn={bn}  →  val_acc={val_acc:.4f}  f1={te_f1:.4f}")

grid_df = pd.DataFrame(grid_results).sort_values('test_f1', ascending=False)
print("\nTop 5:")
print(grid_df.head(5).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
x_labels = [
    f"{r['units']}\ndr={r['dropout']} bn={r['batch_norm']}"
    for _, r in grid_df.iterrows()
]
ax.bar(range(len(grid_df)), grid_df['test_f1'],
       color=sns.color_palette('muted', len(grid_df)), edgecolor='white')
ax.set_xticks(range(len(grid_df)))
ax.set_xticklabels(x_labels, fontsize=6.5, rotation=45, ha='right')
ax.set_ylabel('Test F1 Macro')
ax.set_title('Architecture Grid Search Results')
sns.despine()
plt.tight_layout()
plt.show()

## 9. Optimizer Comparison

In [ ]:
best_row   = grid_df.iloc[0]
best_units = eval(best_row['units'])
best_dr    = best_row['dropout']
best_l2    = best_row['l2']
best_bn    = best_row['batch_norm']

optimizers = {
    'Adam':    keras.optimizers.Adam(learning_rate=5e-4),
    'AdamW':   keras.optimizers.AdamW(learning_rate=5e-4, weight_decay=1e-4),
    'RMSprop': keras.optimizers.RMSprop(learning_rate=5e-4, momentum=0.9),
    'SGD':     keras.optimizers.SGD(learning_rate=1e-2, momentum=0.9, nesterov=True),
}

opt_results = {}

for name, opt in optimizers.items():
    m = build_deep_mlp(n_features, n_classes,
                       units=best_units, dropout=best_dr, l2=best_l2, batch_norm=best_bn)
    m.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])
    h = m.fit(
        ds_train, validation_data=ds_val, epochs=200,
        class_weight=class_weights,
        callbacks=[EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True)],
        verbose=0
    )
    y_pred = np.argmax(m.predict(X_test, verbose=0), axis=1)
    opt_results[name] = {
        'history': h,
        'val_acc': max(h.history['val_accuracy']),
        'test_f1': f1_score(yi_test, y_pred, average='macro')
    }
    print(f"{name:10s}  val_acc={opt_results[name]['val_acc']:.4f}  test_f1={opt_results[name]['test_f1']:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name, res in opt_results.items():
    h = res['history'].history
    axes[0].plot(h['val_loss'],     label=name, linewidth=1.6)
    axes[1].plot(h['val_accuracy'], label=name, linewidth=1.6)

axes[0].set_title('Val Loss by Optimizer')
axes[0].set_xlabel('Epoch')
axes[0].legend(fontsize=8)
axes[1].set_title('Val Accuracy by Optimizer')
axes[1].set_xlabel('Epoch')
axes[1].yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1))
axes[1].legend(fontsize=8)
sns.despine()
plt.tight_layout()
plt.show()

## 10. Stratified K-Fold Cross Validation

In [ ]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

fold_accs, fold_f1s = [], []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y_int)):
    X_f_tr, X_f_val = X[train_idx], X[val_idx]
    Y_f_tr, Y_f_val = Y[train_idx], Y[val_idx]
    yi_f_val        = y_int[val_idx]

    ds_f_tr = (
        tf.data.Dataset.from_tensor_slices((X_f_tr, Y_f_tr))
        .shuffle(512, seed=SEED).map(add_noise, num_parallel_calls=AUTOTUNE)
        .batch(BATCH_SIZE).prefetch(AUTOTUNE)
    )
    ds_f_val = (
        tf.data.Dataset.from_tensor_slices((X_f_val, Y_f_val))
        .batch(BATCH_SIZE).prefetch(AUTOTUNE)
    )

    cw_fold = compute_class_weight('balanced', classes=np.arange(n_classes),
                                    y=y_int[train_idx])

    m = build_deep_mlp(n_features, n_classes,
                       units=best_units, dropout=best_dr, l2=best_l2, batch_norm=best_bn)
    m.compile(
        optimizer=keras.optimizers.Adam(5e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    m.fit(
        ds_f_tr, validation_data=ds_f_val, epochs=200,
        class_weight=dict(enumerate(cw_fold)),
        callbacks=[EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True)],
        verbose=0
    )

    y_pred = np.argmax(m.predict(X_f_val, verbose=0), axis=1)
    acc_f  = accuracy_score(yi_f_val, y_pred)
    f1_f   = f1_score(yi_f_val, y_pred, average='macro')
    fold_accs.append(acc_f)
    fold_f1s.append(f1_f)
    print(f"Fold {fold+1}  acc={acc_f:.4f}  f1={f1_f:.4f}")

print(f"\nMean Acc : {np.mean(fold_accs):.4f} ± {np.std(fold_accs):.4f}")
print(f"Mean F1  : {np.mean(fold_f1s):.4f} ± {np.std(fold_f1s):.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
folds = np.arange(1, 6)
ax.plot(folds, fold_accs, marker='o', label='Accuracy', linewidth=1.8)
ax.plot(folds, fold_f1s,  marker='s', label='F1 Macro', linewidth=1.8, linestyle='--')
ax.axhline(np.mean(fold_accs), color='C0', linestyle=':', alpha=0.6)
ax.axhline(np.mean(fold_f1s),  color='C1', linestyle=':', alpha=0.6)
ax.set_xlabel('Fold')
ax.set_ylabel('Score')
ax.set_title('5-Fold Cross Validation')
ax.set_xticks(folds)
ax.legend()
sns.despine()
plt.tight_layout()
plt.show()

## 11. Best Model — Full Training

In [ ]:
best_model = build_deep_mlp(n_features, n_classes,
                             units=best_units, dropout=best_dr,
                             l2=best_l2, batch_norm=best_bn)

best_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

best_model.summary()
count_params(best_model)

In [ ]:
callbacks_best = [
    EarlyStopping(monitor='val_loss', patience=40, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=15, min_lr=1e-7, verbose=1),
    ModelCheckpoint(
        filepath=os.path.join(CHECKPOINT_DIR, 'best_model.keras'),
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    CSVLogger(os.path.join(LOG_DIR, 'best_model.csv'))
]

history_best = best_model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=500,
    class_weight=class_weights,
    callbacks=callbacks_best,
    verbose=0
)

plot_history(history_best, 'Best Model — Full Training')

In [ ]:
best_acc, best_f1, best_pred, best_prob = evaluate_model(
    best_model, X_test, yi_test, title='Best Model'
)

## 12. Fine-Tuning

### 12.1 Freeze all but the final layer

In [ ]:
ft_model = keras.models.clone_model(best_model)
ft_model.set_weights(best_model.get_weights())

for layer in ft_model.layers[:-1]:
    layer.trainable = False

ft_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Trainable layers:")
for l in ft_model.layers:
    print(f"  {l.name:25s}  trainable={l.trainable}")

In [ ]:
history_ft1 = ft_model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=100,
    class_weight=class_weights,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1)
    ],
    verbose=0
)

plot_history(history_ft1, 'Fine-Tune Phase 1 (head only)')

### 12.2 Unfreeze all — full fine-tune at very low LR

In [ ]:
for layer in ft_model.layers:
    layer.trainable = True

ft_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_ft2 = ft_model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=200,
    class_weight=class_weights,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=30, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-8, verbose=1),
        ModelCheckpoint(
            filepath=os.path.join(CHECKPOINT_DIR, 'finetuned_best.keras'),
            monitor='val_accuracy', save_best_only=True, verbose=0
        )
    ],
    verbose=0
)

plot_history(history_ft2, 'Fine-Tune Phase 2 (all layers)')

In [ ]:
ft_acc, ft_f1, ft_pred, ft_prob = evaluate_model(
    ft_model, X_test, yi_test, title='Fine-Tuned Model'
)

## 13. Prediction Confidence & Calibration

In [ ]:
# Confidence histogram
max_probs = ft_prob.max(axis=1)
correct   = (ft_pred == yi_test)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(max_probs[correct],   bins=20, alpha=0.7, label='Correct',   color='steelblue', edgecolor='white')
axes[0].hist(max_probs[~correct],  bins=20, alpha=0.7, label='Incorrect', color='salmon',    edgecolor='white')
axes[0].set_xlabel('Max Softmax Probability')
axes[0].set_ylabel('Count')
axes[0].set_title('Prediction Confidence')
axes[0].legend()

# Reliability diagram (calibration)
n_bins = 10
bin_edges = np.linspace(0, 1, n_bins + 1)
bin_accs, bin_confs, bin_counts = [], [], []
for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
    mask = (max_probs >= lo) & (max_probs < hi)
    if mask.sum() == 0:
        continue
    bin_accs.append(correct[mask].mean())
    bin_confs.append(max_probs[mask].mean())
    bin_counts.append(mask.sum())

axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Perfect calibration')
axes[1].bar(bin_confs, bin_accs, width=0.08, alpha=0.7, label='Model', color='steelblue', edgecolor='white')
axes[1].set_xlabel('Confidence')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Reliability Diagram')
axes[1].legend()

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Per-class probability distribution
fig, axes = plt.subplots(1, n_classes, figsize=(5 * n_classes, 4))
for i, (cls, ax) in enumerate(zip(class_names, axes)):
    mask = (yi_test == i)
    for j, cls2 in enumerate(class_names):
        ax.hist(ft_prob[mask, j], bins=15, alpha=0.6, label=cls2, edgecolor='white')
    ax.set_title(f'True class: {cls}')
    ax.set_xlabel('Predicted probability')
    ax.legend(fontsize=7)
sns.despine()
fig.suptitle('Per-class Softmax Distributions', fontweight='bold')
plt.tight_layout()
plt.show()

## 14. Overfitting Analysis

In [ ]:
def plot_overfit_comparison(histories, labels, metric='accuracy'):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    val_key   = f'val_{metric}'
    for h, lbl in zip(histories, labels):
        hist = h.history
        axes[0].plot(hist[metric],   label=lbl, linewidth=1.5)
        axes[1].plot(hist[val_key],  label=lbl, linewidth=1.5)

    for ax, title in zip(axes, [f'Train {metric}', f'Val {metric}']):
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.legend(fontsize=8)
        if metric == 'accuracy':
            ax.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1))

    sns.despine()
    plt.tight_layout()
    plt.show()

plot_overfit_comparison(
    [history_base, history_deep, history_res, history_cosine, history_best, history_ft2],
    ['Baseline', 'Deep MLP', 'Residual', 'Cosine LR', 'Best', 'Fine-Tuned']
)

In [ ]:
# Train-val gap at best epoch
def overfit_gap(history):
    best_e = np.argmax(history.history['val_accuracy'])
    tr_acc = history.history['accuracy'][best_e]
    vl_acc = history.history['val_accuracy'][best_e]
    return tr_acc, vl_acc, tr_acc - vl_acc

rows = []
for lbl, h in zip(
    ['Baseline', 'Deep MLP', 'Residual', 'Cosine LR', 'Best', 'Fine-Tuned'],
    [history_base, history_deep, history_res, history_cosine, history_best, history_ft2]
):
    tr, vl, gap = overfit_gap(h)
    rows.append({'Model': lbl, 'Train Acc': round(tr,4), 'Val Acc': round(vl,4), 'Gap': round(gap,4)})

pd.DataFrame(rows).set_index('Model')

## 15. Model Comparison Summary

In [ ]:
all_results = pd.DataFrame([
    {'Model': 'Baseline MLP',   'Test Acc': base_acc,    'Test F1': base_f1},
    {'Model': 'Deep MLP',       'Test Acc': deep_acc,    'Test F1': deep_f1},
    {'Model': 'Residual MLP',   'Test Acc': res_acc,     'Test F1': res_f1},
    {'Model': 'Cosine LR MLP',  'Test Acc': cosine_acc,  'Test F1': cosine_f1},
    {'Model': 'Best (tuned)',    'Test Acc': best_acc,    'Test F1': best_f1},
    {'Model': 'Fine-Tuned',     'Test Acc': ft_acc,      'Test F1': ft_f1},
]).sort_values('Test F1', ascending=False).reset_index(drop=True)

print(all_results.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
pal = sns.color_palette('muted', len(all_results))

for ax, metric in zip(axes, ['Test Acc', 'Test F1']):
    bars = ax.bar(all_results['Model'], all_results[metric], color=pal, edgecolor='white')
    ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=8)
    ax.set_ylim(all_results[metric].min() * 0.97, 1.02)
    ax.set_title(metric)
    ax.tick_params(axis='x', rotation=30)

sns.despine()
fig.suptitle('All NN Models — Test Set', fontweight='bold')
plt.tight_layout()
plt.show()

## 16. Post-Training Quantisation

### 16.1 Save full-precision SavedModel

In [ ]:
ft_model.save('fruit_model_fp32.keras')
print("Saved fp32 model.")

### 16.2 TFLite float32 baseline

In [ ]:
converter_f32 = tf.lite.TFLiteConverter.from_keras_model(ft_model)
tflite_f32    = converter_f32.convert()

with open('fruit_model_f32.tflite', 'wb') as f:
    f.write(tflite_f32)

print(f"Float32 TFLite size: {len(tflite_f32) / 1024:.2f} KB")

### 16.3 Dynamic-range quantisation (int8 weights)

In [ ]:
converter_dyn = tf.lite.TFLiteConverter.from_keras_model(ft_model)
converter_dyn.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_dyn = converter_dyn.convert()

with open('fruit_model_dyn.tflite', 'wb') as f:
    f.write(tflite_dyn)

print(f"Dynamic-range TFLite size: {len(tflite_dyn) / 1024:.2f} KB")

### 16.4 Full integer quantisation (int8 activations + weights)

In [ ]:
def representative_dataset():
    for sample in X_train[:200]:
        yield [sample.reshape(1, -1).astype(np.float32)]

converter_int8 = tf.lite.TFLiteConverter.from_keras_model(ft_model)
converter_int8.optimizations                    = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset           = representative_dataset
converter_int8.target_spec.supported_ops        = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_int8.inference_input_type             = tf.int8
converter_int8.inference_output_type            = tf.int8
tflite_int8 = converter_int8.convert()

with open('fruit_model_int8.tflite', 'wb') as f:
    f.write(tflite_int8)

print(f"Int8 TFLite size: {len(tflite_int8) / 1024:.2f} KB")

### 16.5 Quantisation accuracy evaluation

In [ ]:
def run_tflite(tflite_bytes, X_input, int8_io=False):
    interp = tf.lite.Interpreter(model_content=tflite_bytes)
    interp.allocate_tensors()
    inp_det = interp.get_input_details()[0]
    out_det = interp.get_output_details()[0]

    preds = []
    for row in X_input:
        sample = row.reshape(1, -1)
        if int8_io:
            scale, zp = inp_det['quantization']
            sample = (sample / scale + zp).astype(np.int8)
        interp.set_tensor(inp_det['index'], sample)
        interp.invoke()
        out = interp.get_tensor(out_det['index'])
        if int8_io:
            o_scale, o_zp = out_det['quantization']
            out = (out.astype(np.float32) - o_zp) * o_scale
        preds.append(np.argmax(out))
    return np.array(preds)

preds_f32  = run_tflite(tflite_f32,  X_test)
preds_dyn  = run_tflite(tflite_dyn,  X_test)
preds_int8 = run_tflite(tflite_int8, X_test, int8_io=True)

quant_results = pd.DataFrame([
    {'Format': 'Float32 TFLite', 'Size KB': len(tflite_f32)/1024,
     'Acc': accuracy_score(yi_test, preds_f32), 'F1': f1_score(yi_test, preds_f32, average='macro')},
    {'Format': 'Dynamic-range',  'Size KB': len(tflite_dyn)/1024,
     'Acc': accuracy_score(yi_test, preds_dyn), 'F1': f1_score(yi_test, preds_dyn, average='macro')},
    {'Format': 'Int8 Full',      'Size KB': len(tflite_int8)/1024,
     'Acc': accuracy_score(yi_test, preds_int8), 'F1': f1_score(yi_test, preds_int8, average='macro')},
]).set_index('Format')

print(quant_results.round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
pal = sns.color_palette('muted', 3)

for ax, col, ylabel in zip(axes,
                            ['Size KB', 'Acc', 'F1'],
                            ['Model size (KB)', 'Accuracy', 'F1 Macro']):
    bars = ax.bar(quant_results.index, quant_results[col], color=pal, edgecolor='white')
    ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=8)
    ax.set_title(col)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis='x', rotation=20)

sns.despine()
fig.suptitle('Post-Training Quantisation Comparison', fontweight='bold')
plt.tight_layout()
plt.show()

## 17. Convert Int8 TFLite to C Array (Arduino ready)

In [ ]:
def tflite_to_c_array(tflite_bytes, array_name='fruit_model_int8'):
    hex_vals = ', '.join(f'0x{b:02x}' for b in tflite_bytes)
    lines = []
    lines.append(f'// Auto-generated TFLite int8 model')
    lines.append(f'#pragma once')
    lines.append(f'#include <stdint.h>')
    lines.append(f'')
    lines.append(f'const unsigned int   {array_name}_len = {len(tflite_bytes)};')
    # wrap hex values at 80 chars
    chunks = [hex_vals[i:i+720] for i in range(0, len(hex_vals), 720)]
    body   = ',\n  '.join(chunks)
    lines.append(f'const unsigned char  {array_name}_data[] = {{')
    lines.append(f'  {body}')
    lines.append(f'}};')
    return '\n'.join(lines)

c_array = tflite_to_c_array(tflite_int8)

with open('fruit_model_int8.h', 'w') as f:
    f.write(c_array)

print(f"Written fruit_model_int8.h  ({len(c_array):,} chars)")
print(c_array[:500], '\n... (truncated)')

## 18. Final Test Evaluation

In [ ]:
print("=" * 60)
print("FINAL HELD-OUT TEST RESULTS")
print("=" * 60)
_, _, _, _ = evaluate_model(ft_model, X_test, yi_test, title='Fine-Tuned (Keras fp32)')

print("\nQuantised int8 — TFLite inference:")
print(classification_report(yi_test, preds_int8, target_names=class_names))

In [ ]:
# Side-by-side confusion: fp32 Keras vs int8 TFLite
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, preds, title in zip(axes,
                             [ft_pred, preds_int8],
                             ['Fine-Tuned fp32', 'Int8 TFLite']):
    cm = confusion_matrix(yi_test, preds)
    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title)

plt.tight_layout()
plt.show()

## 19. Arduino Sketch Reference (TFLite Micro)

Copy `fruit_model_int8.h` into your Arduino sketch folder.

```cpp
#include <Arduino_APDS9960.h>
#include <TensorFlowLite.h>
#include "tensorflow/lite/micro/all_ops_resolver.h"
#include "tensorflow/lite/micro/micro_interpreter.h"
#include "tensorflow/lite/schema/schema_generated.h"
#include "fruit_model_int8.h"

const char* CLASS_NAMES[] = { /* same order as training */ };
constexpr int TENSOR_ARENA_SIZE = 8 * 1024;
uint8_t tensor_arena[TENSOR_ARENA_SIZE];

tflite::AllOpsResolver    resolver;
const tflite::Model*      model;
tflite::MicroInterpreter* interpreter;
TfLiteTensor*             input;
TfLiteTensor*             output;
tinyml4all::APDS9960      sensor;

// Scale params — copy from scaler.data_min_ / scale_range in Python
const float SCALE_MIN[]   = { /* r_min,  g_min,  b_min  */ };
const float SCALE_RANGE[] = { /* r_range,g_range,b_range */ };

void setup() {
    Serial.begin(115200);
    sensor.begin();
    model       = tflite::GetModel(fruit_model_int8_data);
    interpreter = new tflite::MicroInterpreter(model, resolver, tensor_arena,
                                               TENSOR_ARENA_SIZE);
    interpreter->AllocateTensors();
    input  = interpreter->input(0);
    output = interpreter->output(0);
}

void loop() {
    sensor.readColor();
    float raw[3] = { (float)sensor.r, (float)sensor.g, (float)sensor.b };

    // quantise input
    float in_scale = input->params.scale;
    int   in_zp    = input->params.zero_point;
    for (int i = 0; i < 3; i++) {
        float norm = (raw[i] - SCALE_MIN[i]) / SCALE_RANGE[i];
        input->data.int8[i] = (int8_t)(norm / in_scale + in_zp);
    }

    interpreter->Invoke();

    // de-quantise output
    float out_scale = output->params.scale;
    int   out_zp    = output->params.zero_point;
    int   best = 0;
    float best_p = -1e9;
    for (int c = 0; c < 3; c++) {
        float p = (output->data.int8[c] - out_zp) * out_scale;
        if (p > best_p) { best_p = p; best = c; }
    }

    Serial.print("Detected: ");
    Serial.println(CLASS_NAMES[best]);
    delay(1000);
}
```